# 🐦 Tweet Sentiment Analyzer
### B.Tech CSE (Data Science) — Bennett University
**By: Vasu Singhal**

---

## Project Overview
This notebook walks through the **complete pipeline** of our Tweet Sentiment Analyzer:
1. Understanding the Dataset
2. Exploratory Data Analysis (EDA)
3. VADER Sentiment Analysis
4. Model Evaluation
5. Visualization of Results
6. Flask Web App Integration

---

## Step 1: Import Libraries

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import random
import warnings
warnings.filterwarnings('ignore')

# NLP
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Display settings
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('✅ All libraries imported successfully!')

## Step 2: Create the Tweet Dataset
Since the real Twitter API requires paid access, we simulate realistic tweet data using domain-specific templates. This is a standard approach in NLP research.

In [ ]:
TEMPLATES = {
    'positive': [
        'Just tried {kw} and honestly blown away. So much better than expected!',
        'Can\'t stop talking about {kw} — it\'s a total game changer this year.',
        '{kw} just solved a problem I\'ve had for years. Highly recommend it.',
        'Massive shoutout to the {kw} team. This is incredible work.',
        '{kw} actually exceeded all my expectations. Brilliant stuff.',
        'Using {kw} every day now. Life is genuinely better because of it.',
        'The latest {kw} update is exactly what we needed. Love it.',
        '{kw} is the best thing to happen in tech this year. Amazing work.',
    ],
    'negative': [
        'Really disappointed with {kw} lately. The quality has gone downhill badly.',
        'Spent 2 hours dealing with {kw} issues today. Absolutely frustrating.',
        '{kw} keeps crashing on me. Not acceptable at this point at all.',
        'Why is {kw} still so slow? Someone needs to fix this already.',
        '{kw} was great before. Now it\'s just terrible. What happened?',
        'Wasted my money on {kw}. Horrible experience from start to finish.',
    ],
    'neutral': [
        '{kw} just released a new version. Here is what changed in this update.',
        'Researchers published a new study on the impact of {kw} today.',
        '{kw} is trending globally on social media right now.',
        'According to latest reports, {kw} usage has grown significantly.',
        '{kw} is now available across multiple platforms worldwide.',
    ]
}

def generate_tweets(keyword, pos=8, neg=5, neu=5):
    tweets = []
    for cat, n in [('positive', pos), ('negative', neg), ('neutral', neu)]:
        for i in range(n):
            text = TEMPLATES[cat][i % len(TEMPLATES[cat])].replace('{kw}', keyword)
            tweets.append({'text': text, 'expected_sentiment': cat.capitalize()})
    random.shuffle(tweets)
    return pd.DataFrame(tweets)

# Generate dataset for keyword 'ChatGPT'
keyword = 'ChatGPT'
df = generate_tweets(keyword)
print(f'✅ Dataset created: {len(df)} tweets for keyword: "{keyword}"')
print(f'\nShape: {df.shape}')
df.head(10)

## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
print('=== DATASET OVERVIEW ===')
print(f'Total Tweets     : {len(df)}')
print(f'Columns          : {list(df.columns)}')
print(f'Missing Values   : {df.isnull().sum().sum()}')
print()
print('=== SENTIMENT DISTRIBUTION ===')
print(df['expected_sentiment'].value_counts())
print()
print('=== TWEET LENGTH STATS ===')
df['tweet_length'] = df['text'].apply(len)
print(df['tweet_length'].describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Sentiment Distribution
colors = ['#4ade80', '#f87171', '#94a3b8']
sentiment_counts = df['expected_sentiment'].value_counts()
axes[0].pie(sentiment_counts.values, labels=sentiment_counts.index,
            colors=colors, autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title(f'Sentiment Distribution for "{keyword}"', fontsize=13, fontweight='bold')

# Plot 2: Tweet Length by Sentiment
sns.boxplot(data=df, x='expected_sentiment', y='tweet_length',
            palette={'Positive': '#4ade80', 'Negative': '#f87171', 'Neutral': '#94a3b8'},
            ax=axes[1])
axes[1].set_title('Tweet Length by Sentiment', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Sentiment')
axes[1].set_ylabel('Character Count')

plt.tight_layout()
plt.savefig('eda_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA charts saved as eda_analysis.png')

## Step 4: VADER Sentiment Analysis

**What is VADER?**
- VADER = **V**alence **A**ware **D**ictionary and s**E**ntiment **R**easoner
- Rule-based NLP model specifically designed for **social media text**
- Returns 4 scores: `pos`, `neg`, `neu`, `compound`
- `compound` score: -1.0 (most negative) to +1.0 (most positive)

**Classification Rule:**
```
compound >= 0.05  → Positive
compound <= -0.05 → Negative
else              → Neutral
```

In [ ]:
analyzer = SentimentIntensityAnalyzer()

def analyze_sentiment(text):
    scores = analyzer.polarity_scores(text)
    compound = scores['compound']
    if compound >= 0.05:
        label = 'Positive'
    elif compound <= -0.05:
        label = 'Negative'
    else:
        label = 'Neutral'
    confidence = round(abs(compound) * 0.5 + 0.5, 3)
    return pd.Series({
        'vader_label':    label,
        'compound_score': round(compound, 3),
        'pos_score':      round(scores['pos'], 3),
        'neg_score':      round(scores['neg'], 3),
        'neu_score':      round(scores['neu'], 3),
        'confidence':     confidence
    })

# Apply VADER to all tweets
df[['vader_label', 'compound_score', 'pos_score',
    'neg_score', 'neu_score', 'confidence']] = df['text'].apply(analyze_sentiment)

print('✅ VADER analysis complete!')
print()
print('Sample results:')
df[['text', 'vader_label', 'compound_score', 'confidence']].head(8)

## Step 5: Model Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_true = df['expected_sentiment']
y_pred = df['vader_label']

acc = accuracy_score(y_true, y_pred)
print(f'VADER Accuracy: {acc*100:.2f}%')
print()
print('Classification Report:')
print(classification_report(y_true, y_pred))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred, labels=['Positive', 'Negative', 'Neutral'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Positive', 'Negative', 'Neutral'],
            yticklabels=['Positive', 'Negative', 'Neutral'], ax=axes[0])
axes[0].set_title('Confusion Matrix', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# Compound Score Distribution
colors_map = {'Positive': '#4ade80', 'Negative': '#f87171', 'Neutral': '#94a3b8'}
for label, color in colors_map.items():
    subset = df[df['vader_label'] == label]['compound_score']
    axes[1].hist(subset, alpha=0.6, color=color, label=label, bins=10)
axes[1].set_title('Compound Score Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Compound Score')
axes[1].set_ylabel('Frequency')
axes[1].legend()
axes[1].axvline(x=0.05, color='green', linestyle='--', alpha=0.7, label='Positive threshold')
axes[1].axvline(x=-0.05, color='red', linestyle='--', alpha=0.7, label='Negative threshold')

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Evaluation charts saved as model_evaluation.png')

## Step 6: Final Results & Insights

In [ ]:
counts = df['vader_label'].value_counts()
dominant = counts.idxmax()
pct = round(counts[dominant] / len(df) * 100)
avg_conf = round(df['confidence'].mean() * 100, 1)

print('=' * 50)
print(f'  FINAL RESULTS FOR: "{keyword}"')
print('=' * 50)
print(f'  Total Tweets Analyzed : {len(df)}')
print(f'  Positive              : {counts.get("Positive", 0)}')
print(f'  Negative              : {counts.get("Negative", 0)}')
print(f'  Neutral               : {counts.get("Neutral", 0)}')
print(f'  Overall Mood          : {dominant}')
print(f'  Dominant Percentage   : {pct}%')
print(f'  Avg Confidence        : {avg_conf}%')
print('=' * 50)

# Plain English explanation
explain = {
    'Positive': f'People are talking about "{keyword}" positively — {pct}% of tweets show appreciation or satisfaction.',
    'Negative': f'The conversation around "{keyword}" leans negative — {pct}% of tweets express frustration or disappointment.',
    'Neutral':  f'Tweets about "{keyword}" are mostly informational — {pct}% are neutral in tone.'
}
print(f'\n💬 Summary: {explain[dominant]}')

## Step 7: Flask Web App

The same pipeline above is wrapped in a **Flask web application** for easy use.

### How to run:
```
# Option 1: Double click run.bat (Windows)

# Option 2: Terminal
python app.py
```

Then open: **http://localhost:5000**

### Architecture:
```
User types keyword
      ↓
Flask receives POST /analyze
      ↓
Generate simulated tweets
      ↓
VADER analyzes each tweet
      ↓
Return JSON with scores
      ↓
Chart.js renders 3 charts
      ↓
Plain-English summary shown
```

In [ ]:
print('To launch the web app, run this in terminal:')
print()
print('  python app.py')
print()
print('Or double-click run.bat on Windows')
print()
print('The browser will open automatically at http://localhost:5000')